# Word2Vec

Word2Vec is a family of shallow neural-network models that learn word embeddings by predicting words from their context or context from a word. It was introduced by Tomas Mikolov and colleagues at Google and became one of the most influential methods for learning dense word representations.

## Core idea
- Word2Vec is built on the distributional hypothesis:

    - Words that appear in similar contexts tend to have similar meanings.

- Instead of hand-designing features, Word2Vec learns a vector for each word by training a simple neural network on a self-supervised task over a large text corpus. The learned weights of that network become the word embeddings.

## Two architectures: CBOW and Skip-gram
Word2Vec has two main training objectives. Both use a sliding context window over the text.

### CBOW (Continuous Bag of Words)


-  Input: surrounding context words

- Target: the center (target) word

- Example sentence: I enjoy learning natural language processing
    - With a window size of 2 and target learning:

    - Context words: I, enjoy, natural, language

- CBOW tries to predict learning from these four words.

- CBOW can be thought of as a “fill-in-the-blank” task. It tends to be faster to train and works well for frequent words.

### Skip-gram

- Input: the center (target) word

- Target: surrounding context words

- Same example, target learning, window size 2:

    - Input: learning

    - Targets: I, enjoy, natural, language

- Skip-gram tries to predict multiple context words from a single target. It is often better for rare words and can capture finer-grained relationships, but it is usually slower to train than CBOW.

## Model structure (conceptual)
- At a high level, a Word2Vec model has:

    - An input vocabulary (one-hot encoded word indices)

    - An embedding matrix W (input embeddings)

    - A context/output embedding matrix W′

    - A small hidden layer that is essentially a lookup + optional averaging

    - An output layer that predicts either the target word (CBOW) or context words (Skip-gram) using a softmax over the vocabulary.

- For a target word Wc and a context word Wo, the probability of predicting Wo given Wc is often modeled as:

    > 𝑃( W𝑜 ∣ W𝑐 ) = exp( 𝑢(transpose)W𝑐 X 𝑣(W𝑜)) / ∑(𝑤 ∈ 𝑉) exp( 𝑢(transpose)W𝑐 X 𝑣(W) )

    - where u(Wc) is the input embedding for the center word, 𝑣(W𝑜) is the output embedding for the context word, and V is the vocabulary. The training objective is to maximize the likelihood of observed (target, context) pairs.

- Because the full softmax over a large vocabulary is expensive, practical implementations use tricks such as:

    - **Negative sampling**: instead of updating all output weights, update a small number of “negative” (random) words plus the true context word.

    - **Hierarchical softmax**: organize the output vocabulary as a binary tree to reduce computation.

## Training data: skip-grams
- Even though the method is called “Word2Vec,” it is trained on skip-grams, which are pairs of words that appear within a fixed window, allowing some words to be skipped. For a sentence and window size k, you slide over the text and generate pairs like:

    - (target_word, context_word)

- For example, with window size 2: 'I enjoy learning natural language processing'

    - Pairs (partial):
    - (learning, I)
    - (learning, enjoy)
    - (learning, natural)
    - (learning, language)
    - (enjoy, I)
    - (enjoy, learning)
    - (natural, enjoy)
    - (natural, learning)
    - ...

- These pairs form the training examples for the neural network.



## What you get after training
- After training, you typically keep the input embedding matrix as your word vectors. Each row corresponds to a word in the vocabulary:

    - king   → [0.21, -0.34, 0.08, ..., 0.12]
    - queen  → [0.20, -0.33, 0.09, ..., 0.11]
    - man    → [-0.10, 0.40, -0.05, ..., -0.02]
    - woman  → [-0.09, 0.41, -0.04, ..., -0.01]

- These vectors can then be used in downstream tasks such as classification, clustering, similarity search, or as inputs to sequence models.

## Example relationships

- Well-trained Word2Vec embeddings often exhibit interesting geometric properties:

    - Similar words have high cosine similarity: cat is close to dog, car to automobile.

    - Some linear relationships approximately hold, such as: `king − man + woman ≈ queen`

- This is not perfect and depends heavily on training data and hyperparameters, but it illustrates that the embedding space can encode certain relational patterns.

## Advantages

- **Dense, low-dimensional vectors**: replaces huge sparse one-hot vectors with compact embeddings (e.g., 100–300 dimensions).

- **Captures semantic and syntactic similarity**: words with similar usage patterns end up close in vector space.

- **Computationally efficient** compared to deeper models, especially with negative sampling or hierarchical softmax.

- **Widely used and available**: many pretrained Word2Vec models exist for various languages and domains.

## Limitations

- **Static embeddings**: each word has one fixed vector, regardless of context. It cannot distinguish different senses of bank (financial vs river).

- **Out-of-vocabulary problem**: unseen words at test time have no vector unless you use extensions like FastText.

- **No subword information**: standard Word2Vec treats words as atomic units and ignores morphology.

- **Fixed context window**: captures only local context, not long-range dependencies.

- **Training-data bias**: embeddings can reflect and amplify biases present in the corpus.

- **Domain dependence**: embeddings trained on general web text may not suit specialized domains like biomedical, legal, or cybersecurity text without retraining.

## CBOW vs Skip-gram: quick comparison

| Aspect               | CBOW                                              | Skip-gram                                      |
| -------------------- | ------------------------------------------------- | ---------------------------------------------- |
| Prediction direction | Predict target from context                       | Predict context from target                    |
| Speed                | Generally faster                                  | Generally slower                               |
| Rare words           | Less effective                                    | Often better                                   |
| Typical use          | Large datasets with frequent words, speed matters | Smaller or noisier datasets, rare words matter |

Both are part of the Word2Vec family and are commonly implemented together in libraries.